# LLM Alignment Techniques: Beyond RLHF and DPO

Alignment is the process of training language models to follow human instructions, be helpful, harmless, and honest.
The field has rapidly evolved from the original PPO-based RLHF pipeline to a rich ecosystem of preference-learning methods.

## Overview of Alignment Methods

| Method | Reference Model Needed | Data Format | Key Innovation |
|--------|----------------------|-------------|----------------|
| **RLHF** | Yes (reward model + ref policy) | Preference pairs → PPO rollouts | First scalable alignment via RL |
| **DPO** | Yes (frozen ref policy) | (prompt, chosen, rejected) | Closed-form RL; eliminates separate reward model |
| **IPO** | Yes (frozen ref policy) | (prompt, chosen, rejected) | Regularization prevents preference overfitting |
| **KTO** | Yes (frozen ref policy) | (prompt, completion, label) | Single labels; prospect theory value function |
| **ORPO** | No | (prompt, chosen, rejected) | Combined SFT + preference in one loss |
| **SimPO** | No | (prompt, chosen, rejected) | Length-normalized reward + target margin |
| **RLAIF** | Yes (LLM-as-judge) | Prompts only (LLM labels) | Replaces human raters with AI feedback |
| **Constitutional AI** | Yes (critique model) | Principles list + prompts | Self-critique and revision loop |
| **SPIN** | Yes (previous iteration) | SFT dataset only | Self-play: model vs its own previous version |
| **Rejection Sampling** | Yes (reward model) | SFT dataset | Best-of-N at train time; inference-time scaling |
| **Iterative DPO** | Yes (frozen ref policy) | Online rollouts | Fresh preference data generated during training |

## Why Alignment Matters

Pretrained LLMs are trained to predict the next token in internet text they learn to mimic
human writing, including harmful content. Alignment transforms a raw language model into an
assistant that:

- Follows instructions reliably
- Refuses harmful requests
- Is calibrated and honest
- Produces preferred output style and length

The core challenge: **defining and measuring "good" behavior** so it can be optimized.


## RLHF: The Foundation

Reinforcement Learning from Human Feedback (RLHF), introduced in InstructGPT (Ouyang et al., 2022),
established the blueprint for aligning large language models. It has three stages:

### Stage 1: Supervised Fine-Tuning (SFT)

A pretrained model is fine-tuned on a curated set of (prompt, demonstration) pairs where human
contractors write ideal responses. This creates the SFT policy $\pi_{SFT}$.

### Stage 2: Reward Model Training

Human raters compare pairs of model outputs $(y_1, y_2)$ for the same prompt $x$.
A reward model $r_\phi$ is trained to predict human preferences using the
**Bradley-Terry** model of pairwise comparisons:

$$P(y_w \succ y_l \mid x) = \sigma(r_\phi(x, y_w) - r_\phi(x, y_l))$$

The reward model loss (negative log-likelihood of human preference labels):

$$\mathcal{L}_{RM}(r_\phi) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}\left[\log \sigma\left(r_\phi(x, y_w) - r_\phi(x, y_l)\right)\right]$$

where $y_w$ is the preferred ("winner") response and $y_l$ is the dispreferred ("loser") response.

### Stage 3: PPO Fine-Tuning

The SFT model is further trained using **Proximal Policy Optimization (PPO)** to maximize
the reward model score while staying close to the SFT policy via a KL penalty:

$$\text{objective}(\pi_\theta) = \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi_\theta(\cdot|x)}\left[r_\phi(x, y) - \beta \cdot \text{KL}\left[\pi_\theta(\cdot|x) \| \pi_{SFT}(\cdot|x)\right]\right]$$

The **KL penalty** $\beta$ (typically 0.01-0.1) prevents the model from exploiting the reward
model by generating gibberish that scores high but is meaningless a phenomenon called
**reward hacking**.

### RLHF Challenges

| Challenge | Description |
|-----------|-------------|
| Training complexity | Requires 3 separate model training stages |
| Memory overhead | Policy + reference + reward model all in GPU memory |
| Reward hacking | Model exploits reward model errors |
| Human labeling cost | Preference data is expensive to collect |
| PPO instability | Sensitive to hyperparameters; can diverge |


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class BradleyTerryRewardModel(nn.Module):
    """
    Reward model based on the Bradley-Terry comparison model.
    Takes a hidden state (e.g. last token embedding from a transformer)
    and outputs a scalar reward score.
    """

    def __init__(self, hidden_size: int = 768, dropout: float = 0.1):
        super().__init__()
        self.reward_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """
        Args:
            hidden_states: [batch_size, hidden_size] last token representation
        Returns:
            rewards: [batch_size] scalar reward per sequence
        """
        return self.reward_head(hidden_states).squeeze(-1)

    def preference_loss(
        self,
        chosen_hidden: torch.Tensor,
        rejected_hidden: torch.Tensor,
    ) -> tuple[torch.Tensor, dict]:
        """
        Bradley-Terry pairwise preference loss.

        Loss = -E[log sigma(r(x, y_w) - r(x, y_l))]

        Args:
            chosen_hidden:   [batch, hidden] last-token hidden state for preferred responses
            rejected_hidden: [batch, hidden] last-token hidden state for dispreferred responses
        Returns:
            loss:   scalar cross-entropy loss
            metrics: dict with reward statistics
        """
        r_chosen = self.forward(chosen_hidden)    # [batch]
        r_rejected = self.forward(rejected_hidden)  # [batch]

        # Bradley-Terry loss: -log sigma(r_w - r_l)
        loss = -F.logsigmoid(r_chosen - r_rejected).mean()

        # Accuracy: fraction of pairs where r_w > r_l
        accuracy = (r_chosen > r_rejected).float().mean()

        metrics = {
            "reward_chosen_mean": r_chosen.mean().item(),
            "reward_rejected_mean": r_rejected.mean().item(),
            "reward_margin": (r_chosen - r_rejected).mean().item(),
            "accuracy": accuracy.item(),
        }
        return loss, metrics


# ------------------------------------------------------------------
# Sample forward pass
# ------------------------------------------------------------------
torch.manual_seed(42)

hidden_size = 768
batch_size = 4

reward_model = BradleyTerryRewardModel(hidden_size=hidden_size)

# Simulate last-token hidden states for chosen and rejected responses
# In practice these come from the LLM backbone
chosen_hidden = torch.randn(batch_size, hidden_size)
rejected_hidden = torch.randn(batch_size, hidden_size)

# Individual reward scores
r_chosen = reward_model(chosen_hidden)
r_rejected = reward_model(rejected_hidden)
print("Reward scores (chosen):  ", r_chosen.detach().tolist())
print("Reward scores (rejected):", r_rejected.detach().tolist())

# Preference loss
loss, metrics = reward_model.preference_loss(chosen_hidden, rejected_hidden)
print(f"\nBradley-Terry Loss:      {loss.item():.4f}")
print(f"Mean reward margin:      {metrics['reward_margin']:.4f}")
print(f"Pairwise accuracy:       {metrics['accuracy']:.2%}")

# Parameter count
total_params = sum(p.numel() for p in reward_model.parameters())
print(f"\nReward head parameters:  {total_params:,}")


Reward scores (chosen):   [-0.2063113898038864, -0.35295695066452026, -0.09723672270774841, 0.07360319793224335]
Reward scores (rejected): [-0.18504898250102997, -0.11697141081094742, -0.1569264531135559, 0.08329565823078156]

Bradley-Terry Loss:      0.7465
Mean reward margin:      -0.0935
Pairwise accuracy:       50.00%

Reward head parameters:  295,681


## DPO: Direct Preference Optimization

DPO (Rafailov et al., 2023) eliminates the reward model and PPO loop entirely.
The key insight is that the **optimal policy under KL-constrained RL can be written in closed form**,
so we can reparameterize the reward directly in terms of the policy:

$$r^*(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta \log Z(x)$$

Substituting this implicit reward into the Bradley-Terry preference model and simplifying,
the partition function $Z(x)$ cancels and we get the **DPO loss**:

$$\mathcal{L}_{DPO}(\pi_\theta) = -\mathbb{E}_{(x,y_w,y_l)}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right]$$

### Interpretation

The term $\log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$ is the **implicit log-reward** that the policy
assigns to response $y$ relative to the reference. DPO pushes the policy to:

- **Increase** the implicit reward for chosen responses: $\pi_\theta(y_w|x) \uparrow$ relative to $\pi_{ref}$
- **Decrease** the implicit reward for rejected responses: $\pi_\theta(y_l|x) \downarrow$ relative to $\pi_{ref}$

### The $\beta$ Hyperparameter

| $\beta$ value | Effect |
|---------------|--------|
| $\beta \to 0$ | No KL constraint; policy can collapse or drift arbitrarily |
| $\beta \approx 0.01$-$0.1$ | Mild constraint; some preference learning with low alignment tax |
| $\beta \approx 0.5$-$1.0$ | Strong constraint; policy stays close to reference, less change |
| $\beta \to \infty$ | Policy equals reference; no alignment signal |

### DPO vs RLHF

| Property | RLHF (PPO) | DPO |
|----------|-----------|-----|
| Reward model | Separate training required | Implicit; no separate model |
| Training stages | 3 (SFT → RM → PPO) | 2 (SFT → DPO) |
| Memory | Policy + ref + reward + critic | Policy + ref |
| Stability | PPO can diverge | Simple cross-entropy; stable |
| Online/Offline | Online (requires rollouts) | Offline (static dataset) |
| Reward hacking | Moderated by KL in PPO | Still possible via OOD responses |

### Alignment Tax

The **alignment tax** refers to the capability degradation that can occur when aligning a model.
DPO can decrease log-probabilities of both chosen and rejected completions, potentially causing
**likelihood displacement** the model assigns mass to unexpected tokens rather than improving
on preferred responses. This is an active area of research.


## IPO: Identity Preference Optimization

IPO (Azar et al., 2023) identifies a critical failure mode in DPO: when the dataset is deterministic
(each prompt maps to a unique pair), DPO can **overfit to the pairwise ranking** and push
log-probabilities to $\pm \infty$, losing the KL regularization guarantee.

### The DPO Overfitting Problem

DPO's loss can be made arbitrarily small by increasing the log-ratio margin:

$$\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} \to +\infty$$

This causes the policy to become deterministic on the training set, which breaks the
probabilistic interpretation and may harm generalization.

### IPO Loss

IPO replaces the log-sigmoid with a **squared loss** that directly regularizes the margin:

$$\mathcal{L}_{IPO}(\pi_\theta) = \mathbb{E}_{(x,y_w,y_l)}\left[\left(\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} - \frac{1}{2\beta}\right)^2\right]$$

### Key Differences from DPO

| Aspect | DPO | IPO |
|--------|-----|-----|
| Loss shape | $-\log \sigma(\cdot)$ (unbounded below) | $(\cdot - \frac{1}{2\beta})^2$ (quadratic) |
| Target margin | 0 (just rank correctly) | $\frac{1}{2\beta}$ (explicit target gap) |
| Overfitting | Can overfit to deterministic data | Naturally regularized by squared loss |
| Derivation | Exact RL closed-form | Direct regularized objective |

The optimal solution of IPO is:

$$h^*(x, y_w, y_l) = \log \frac{\pi^*(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi^*(y_l|x)}{\pi_{ref}(y_l|x)} = \frac{1}{2\beta}$$

This means the policy learns to maintain a **fixed, finite margin** between chosen and rejected
responses rather than pushing them apart without bound.

In practice, IPO tends to perform similarly to DPO but shows better behavior when the
preference data is noisy or when training for many epochs.


## KTO: Kahneman-Tversky Optimization

KTO (Ethayarajh et al., 2024) takes a radically different approach: instead of requiring
**paired** preference data (chosen vs. rejected for the same prompt), KTO trains on
**individual** (prompt, completion, desirability_label) examples where the label is simply
binary: "good" or "bad".

### Motivation: Prospect Theory

Human decision-making is well-described by Kahneman and Tversky's **Prospect Theory**:
humans are more sensitive to losses than equivalent gains (loss aversion).
KTO argues the same principle should govern language model alignment the disutility of
generating a bad response should outweigh the utility of generating a good one.

### KTO Implicit Reward

The implicit log-reward is the same as DPO:

$$\hat{r}_\theta(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$$

The **reference point** $z_{ref}$ estimates the expected reward under the reference policy:

$$z_{ref} = \mathbb{E}_{x' \sim \mathcal{D}, y' \sim \pi_{ref}(\cdot|x')}\left[\beta \log \frac{\pi_{ref}(y'|x')}{\pi_{ref}(y'|x')}\right] = 0$$

In practice, $z_{ref}$ is estimated from the current batch using a KL term.

### KTO Loss

$$\mathcal{L}_{KTO} = \mathbb{E}_{(x,y) \sim \mathcal{D}}\left[v\left(\hat{r}_\theta(x, y) - z_{ref}\right)\right]$$

where the **value function** from prospect theory is:

$$v(u) = \begin{cases} \lambda_D \cdot \sigma(u) & \text{if } y \text{ is desirable} \\ \lambda_U \cdot (1 - \sigma(u)) & \text{if } y \text{ is undesirable} \end{cases}$$

where $\lambda_D$ and $\lambda_U$ control the weighting of gains vs losses ($\lambda_U > \lambda_D$
to model loss aversion).

### Practical Advantages

| Advantage | Details |
|-----------|---------|
| No pairing required | Any (prompt, completion) from SFT data can be used directly |
| Works with existing datasets | Instruction-following datasets with quality labels |
| Handles label imbalance | Loss aversion weighting adjusts for unequal good/bad counts |
| Scales to web data | Internet text can be rated good/bad without pairing |

KTO has shown competitive performance with DPO while requiring less curated preference data,
making it attractive for low-resource alignment scenarios.


## ORPO: Odds Ratio Preference Optimization

ORPO (Hong et al., 2024) eliminates the reference model entirely by incorporating the
preference signal **directly into the SFT loss** as a single-stage training objective.

### Key Insight

Standard SFT maximizes the log-probability of chosen responses but does nothing to discourage
the model from also assigning high probability to rejected responses. ORPO adds a **penalty
term** based on the odds ratio of chosen vs. rejected:

### Odds Ratio

The odds of a response $y$ given prompt $x$ under policy $\pi_\theta$:

$$\text{odds}_\theta(y|x) = \frac{\pi_\theta(y|x)}{1 - \pi_\theta(y|x)}$$

The log-odds ratio of chosen over rejected:

$$\text{OR}_\theta(y_w, y_l | x) = \log \frac{\text{odds}_\theta(y_w|x)}{\text{odds}_\theta(y_l|x)}$$

### ORPO Loss

$$\mathcal{L}_{ORPO} = \mathcal{L}_{SFT} + \lambda \cdot \mathcal{L}_{OR}$$

where:

$$\mathcal{L}_{SFT} = -\mathbb{E}\left[\log \pi_\theta(y_w|x)\right]$$

$$\mathcal{L}_{OR} = -\mathbb{E}\left[\log \sigma\left(\text{OR}_\theta(y_w, y_l | x)\right)\right]$$

### Why No Reference Model?

The SFT loss acts as the "anchor" that plays the role of the reference model.
The odds ratio measures preference relative to the model's **own** current distribution
rather than a frozen reference. This removes the need to:

1. Train a separate SFT model and freeze it
2. Keep a reference model copy in memory
3. Run two forward passes per training step

### ORPO Properties

| Property | Value |
|----------|-------|
| Training stages | 1 (SFT + preference jointly) |
| Reference model | Not required |
| Memory overhead | ~50% less than DPO |
| $\lambda$ typical value | 0.1 |
| Benchmark performance | Competitive with DPO on Alpaca Eval |

ORPO is particularly attractive for fine-tuning under tight resource constraints,
as it reduces both the number of training stages and GPU memory requirements.


## SimPO: Simple Preference Optimization

SimPO (Meng et al., 2024) makes two critical improvements over DPO:

1. **Removes the reference model** by using average log-probability as the implicit reward
2. **Adds a target reward margin** $\gamma$ to improve separation between chosen and rejected

### DPO's Length Bias Problem

DPO computes log-probabilities as sums over tokens. Longer responses naturally have
lower total log-probabilities (more tokens to multiply), creating a **length bias**:
models learn to prefer shorter completions even when longer ones are better.

### SimPO Reward

SimPO defines the reward as the **average** log-probability per token:

$$r_{SimPO}(x, y) = \frac{1}{|y|} \log \pi_\theta(y|x) = \frac{1}{|y|} \sum_{t=1}^{|y|} \log \pi_\theta(y_t | x, y_{<t})$$

This normalizes by response length, removing the length bias.

### SimPO Loss

$$\mathcal{L}_{SimPO}(\pi_\theta) = -\mathbb{E}_{(x,y_w,y_l)}\left[\log \sigma\left(\frac{\beta}{|y_w|}\log\pi_\theta(y_w|x) - \frac{\beta}{|y_l|}\log\pi_\theta(y_l|x) - \gamma\right)\right]$$

The **target margin** $\gamma > 0$ ensures a minimum gap between chosen and rejected rewards.
Without this, the model could satisfy the loss with a tiny margin, failing to clearly
distinguish good from bad responses.

### DPO vs SimPO Comparison

| Aspect | DPO | SimPO |
|--------|-----|-------|
| Reference model | Required | Not required |
| Reward formulation | Log-ratio (policy / reference) | Average log-prob (policy only) |
| Length normalization | No | Yes (per-token average) |
| Target margin | None ($\gamma = 0$) | Explicit $\gamma$ (tuned, e.g. 0.5-2.0) |
| Memory efficiency | Moderate | High (no reference forward pass) |

SimPO achieves state-of-the-art results on AlpacaEval 2.0 and ArenaHard while using
less memory than DPO, making it one of the most efficient alignment methods available.


In [2]:
import torch
import torch.nn.functional as F
from typing import Optional


def compute_dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    reference_chosen_logps: torch.Tensor,
    reference_rejected_logps: torch.Tensor,
    beta: float = 0.1,
) -> tuple[torch.Tensor, dict]:
    """
    Direct Preference Optimization loss.
    L = -E[log sigma(beta * (log pi(y_w|x) - log pi_ref(y_w|x))
                          - beta * (log pi(y_l|x) - log pi_ref(y_l|x)))]

    All logps are summed log-probabilities of the entire response.
    """
    pi_logratios = policy_chosen_logps - policy_rejected_logps
    ref_logratios = reference_chosen_logps - reference_rejected_logps
    logits = beta * (pi_logratios - ref_logratios)
    loss = -F.logsigmoid(logits).mean()

    chosen_rewards = beta * (policy_chosen_logps - reference_chosen_logps).detach()
    rejected_rewards = beta * (policy_rejected_logps - reference_rejected_logps).detach()
    reward_margin = (chosen_rewards - rejected_rewards).mean()
    accuracy = (chosen_rewards > rejected_rewards).float().mean()

    return loss, {"margin": reward_margin.item(), "accuracy": accuracy.item()}


def compute_ipo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    reference_chosen_logps: torch.Tensor,
    reference_rejected_logps: torch.Tensor,
    beta: float = 0.1,
) -> tuple[torch.Tensor, dict]:
    """
    Identity Preference Optimization loss.
    L = E[(log_ratio_chosen - log_ratio_rejected - 1/(2*beta))^2]

    Uses squared loss with explicit target margin 1/(2*beta).
    """
    log_ratio_chosen = policy_chosen_logps - reference_chosen_logps
    log_ratio_rejected = policy_rejected_logps - reference_rejected_logps
    h = log_ratio_chosen - log_ratio_rejected
    target_margin = 1.0 / (2.0 * beta)
    loss = ((h - target_margin) ** 2).mean()

    accuracy = (log_ratio_chosen > log_ratio_rejected).float().mean()
    return loss, {"h_mean": h.mean().item(), "target": target_margin, "accuracy": accuracy.item()}


def compute_orpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    reference_chosen_logps: Optional[torch.Tensor] = None,
    reference_rejected_logps: Optional[torch.Tensor] = None,
    beta: float = 0.1,
    lambda_orpo: float = 0.1,
) -> tuple[torch.Tensor, dict]:
    """
    Odds Ratio Preference Optimization loss (no reference model needed).
    L = L_SFT + lambda * L_OR

    L_SFT = -E[log pi(y_w|x)]                     (maximize chosen log-prob)
    L_OR  = -E[log sigma(log(odds_w / odds_l))]   (log-odds ratio penalty)

    Note: reference_chosen/rejected_logps are ignored (kept for API compatibility).
    """
    # SFT component: maximize chosen log-prob
    l_sft = -policy_chosen_logps.mean()

    # Odds = p / (1 - p); for log-prob: log_odds = log_prob - log(1 - exp(log_prob))
    # Simplified: use log_prob directly as a proxy for log-odds (common approximation)
    log_odds_chosen = policy_chosen_logps - torch.log1p(-policy_chosen_logps.exp().clamp(max=1 - 1e-7))
    log_odds_rejected = policy_rejected_logps - torch.log1p(-policy_rejected_logps.exp().clamp(max=1 - 1e-7))
    log_odds_ratio = log_odds_chosen - log_odds_rejected
    l_or = -F.logsigmoid(log_odds_ratio).mean()

    loss = l_sft + lambda_orpo * l_or
    accuracy = (policy_chosen_logps > policy_rejected_logps).float().mean()
    return loss, {"l_sft": l_sft.item(), "l_or": l_or.item(), "accuracy": accuracy.item()}


# ------------------------------------------------------------------
# Comparison on sample data
# ------------------------------------------------------------------
torch.manual_seed(0)
batch_size = 8

# Simulate sequence-level log-probabilities (sum over tokens)
# Chosen responses should generally have higher policy log-prob than rejected
policy_chosen_logps = torch.randn(batch_size) * 2.0 - 5.0   # e.g. around -5
policy_rejected_logps = torch.randn(batch_size) * 2.0 - 8.0  # e.g. around -8
reference_chosen_logps = torch.randn(batch_size) * 2.0 - 6.0
reference_rejected_logps = torch.randn(batch_size) * 2.0 - 7.0

print("=" * 60)
print("Loss Function Comparison (beta=0.1)")
print("=" * 60)

dpo_loss, dpo_info = compute_dpo_loss(
    policy_chosen_logps, policy_rejected_logps,
    reference_chosen_logps, reference_rejected_logps, beta=0.1
)
ipo_loss, ipo_info = compute_ipo_loss(
    policy_chosen_logps, policy_rejected_logps,
    reference_chosen_logps, reference_rejected_logps, beta=0.1
)
orpo_loss, orpo_info = compute_orpo_loss(
    policy_chosen_logps, policy_rejected_logps, beta=0.1
)

print(f"\nDPO  loss: {dpo_loss.item():.4f}  | margin: {dpo_info['margin']:.4f} | accuracy: {dpo_info['accuracy']:.2%}")
print(f"IPO  loss: {ipo_loss.item():.4f}  | h_mean: {ipo_info['h_mean']:.4f} | target: {ipo_info['target']:.4f}")
print(f"ORPO loss: {orpo_loss.item():.4f}  | L_sft: {orpo_info['l_sft']:.4f} | L_or: {orpo_info['l_or']:.4f}")

# Sensitivity to beta
print("\nDPO loss vs beta (fixed data):")
for b in [0.01, 0.05, 0.1, 0.5, 1.0]:
    l, info = compute_dpo_loss(
        policy_chosen_logps, policy_rejected_logps,
        reference_chosen_logps, reference_rejected_logps, beta=b
    )
    print(f"  beta={b:.2f}  loss={l.item():.4f}  accuracy={info['accuracy']:.2%}")


Loss Function Comparison (beta=0.1)

DPO  loss: 0.6507  | margin: 0.1383 | accuracy: 62.50%
IPO  loss: 32.9427  | h_mean: 1.3827 | target: 5.0000
ORPO loss: 5.4394  | L_sft: 5.4011 | L_or: 0.3822

DPO loss vs beta (fixed data):
  beta=0.01  loss=0.6865  accuracy=62.50%
  beta=0.05  loss=0.6654  accuracy=62.50%
  beta=0.10  loss=0.6507  accuracy=62.50%
  beta=0.50  loss=0.8495  accuracy=62.50%
  beta=1.00  loss=1.3759  accuracy=62.50%


## RLAIF and Constitutional AI

### RLAIF: Reinforcement Learning from AI Feedback

RLAIF (Bai et al., 2022; Lee et al., 2023) replaces human raters with an **LLM-as-judge**
to generate preference labels. A "preference model" LLM is prompted to evaluate pairs of
responses and produce preference probabilities:

```
Prompt: "Which response is more helpful, harmless, and honest?
Response A: ...
Response B: ...
Answer: Response [A/B] because..."
```

The AI-generated preference labels are then used identically to human labels in RLHF/DPO training.

**Key findings:** RLAIF-trained models can match or exceed RLHF models on helpfulness benchmarks,
while being orders of magnitude cheaper (no human labelers required).

### Constitutional AI

Constitutional AI (CAI, Bai et al., 2022) from Anthropic is a two-stage process for
training helpful, harmless, and honest models:

**Stage 1: Supervised Learning from AI Feedback (SL-CAI)**

```
1. Generate initial response to a potentially harmful prompt
2. Ask the model to critique its own response against a principle:
   "Does this response violate the principle: [PRINCIPLE]?"
3. Ask the model to revise the response to better satisfy the principle
4. Repeat critique-revision for multiple principles
5. Fine-tune on the final revised responses
```

**Stage 2: RL from AI Feedback (RLAIF-CAI)**

```
1. Sample pairs of responses to harmful prompts
2. Ask the model: "Which response is less harmful according to [PRINCIPLE]?"
3. Use the AI preference labels to train a preference model (PM)
4. Run PPO or DPO using the PM as reward signal
```

### Example Constitutional Principles

| Principle Category | Example Principle |
|--------------------|------------------|
| Harmlessness | "Choose the response least likely to be used by a malicious actor" |
| Honesty | "Choose the response that is most truthful and does not mislead" |
| Ethics | "Which response better supports human rights and democratic values?" |
| Helpfulness | "Which response is more helpful, informative, and actionable?" |

### Self-Improvement Loop

```
Initial SFT Model
      |
      v
Generate harmful response
      |
      v
Critique ("This violates principle X because...")
      |
      v
Revise ("Here is a revised response that does not...")
      |
      v
Fine-tune on revised responses
      |
      v
Better SFT Model  --> Feed back to AI Feedback labeling --> RLAIF
```

CAI enables scaling alignment to new capability domains without requiring specialized human
expertise for each domain the model's own knowledge is used to guide improvement.


## SPIN: Self-Play Fine-Tuning

SPIN (Self-Play Fine-tuning, Chen et al., 2024) frames alignment as a **two-player game**:
the current model ("main player") tries to generate responses indistinguishable from the
ground-truth SFT data, while a "weak opponent" (the previous iteration's model) generates
fake responses.

### The Self-Play Framework

Given only the SFT dataset $\mathcal{D} = \{(x_i, y_i)\}$ (no additional preference labels needed):

**Iteration $t$:**
1. Use $\pi_\theta^{(t-1)}$ as the "opponent" to generate synthetic rejected responses: $\tilde{y} \sim \pi_\theta^{(t-1)}(\cdot|x)$
2. Construct preference pairs: $(x, y^*, \tilde{y})$ where $y^*$ is the SFT ground-truth
3. Train $\pi_\theta^{(t)}$ with DPO-style loss on these pairs:

$$\mathcal{L}_{SPIN}^{(t)}(\pi_\theta) = -\mathbb{E}\left[\log \sigma\left(\lambda \log \frac{\pi_\theta(y^*|x)}{\pi_\theta^{(t-1)}(y^*|x)} - \lambda \log \frac{\pi_\theta(\tilde{y}|x)}{\pi_\theta^{(t-1)}(\tilde{y}|x)}\right)\right]$$

### Convergence Property

SPIN is guaranteed to converge when the main player's policy equals the data distribution:
$\pi_\theta = p_{data}$. At this point, the model cannot be distinguished from ground-truth
responses, and the loss saturates. This provides a **theoretically grounded stopping criterion**.

### SPPO: Self-Play Preference Optimization

SPPO (Wu et al., 2024) extends SPIN using a Nash equilibrium objective:

$$\mathcal{L}_{SPPO}(\pi_\theta) = \mathbb{E}_{x, y_w, y_l}\left[\left(\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} + \frac{1}{2}\right)^2 + \left(\log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} - \frac{1}{2}\right)^2\right]$$

SPPO iteratively updates the reference policy with each training iteration, enabling
online self-play without requiring human preference labels.

### SPIN vs Standard DPO

| Property | DPO | SPIN |
|----------|-----|------|
| Preference data needed | Yes (human or AI labels) | No (SFT data only) |
| Opponent | Frozen SFT model | Previous training iteration |
| Self-improvement | No static reference | Yes iteratively stronger opponent |
| Convergence guarantee | None specified | Provable under mild conditions |

SPIN demonstrated that **a model can improve beyond SFT quality using only SFT data**,
which was a surprising and important finding for the field.


## Rejection Sampling & Best-of-N

Rejection sampling (also called Best-of-N or BoN) is one of the simplest and most effective
alignment techniques: generate many candidate responses and keep only the best one according
to a reward model.

### Best-of-N Sampling

For a given prompt $x$:
1. Sample $N$ independent completions: $y_1, y_2, \ldots, y_N \sim \pi(\cdot|x)$
2. Score each with a reward model: $r_i = r_\phi(x, y_i)$
3. Return the highest-scored completion: $\hat{y} = y_{\arg\max_i r_i}$

### Expected Reward Improvement

If rewards are i.i.d. with CDF $F(r)$, the expected maximum reward over $N$ samples is:

$$\mathbb{E}[r_{\max}^{(N)}] = \int_{-\infty}^{\infty} r \cdot N \cdot F(r)^{N-1} f(r) \, dr$$

For normally distributed rewards $r \sim \mathcal{N}(\mu, \sigma^2)$, the expected maximum scales approximately as:

$$\mathbb{E}[r_{\max}^{(N)}] \approx \mu + \sigma \cdot \Phi^{-1}\left(1 - \frac{1}{N}\right)$$

where $\Phi^{-1}$ is the inverse normal CDF. This grows as $\mathcal{O}(\sqrt{\log N})$.

### KL Divergence of Best-of-N

The BoN policy has a known KL divergence from the base policy:

$$\text{KL}[\pi_{BoN} \| \pi] \approx \log N - \frac{N-1}{N}$$

This means BoN with $N = e^k$ samples costs approximately $k$ nats of KL.

### Rejection Sampling Fine-Tuning (RFT)

Rejection sampling can also be used to **generate better training data**:

1. For each training prompt, generate $N$ completions
2. Score with a reward model and keep only those above threshold $\tau$
3. Fine-tune on the filtered high-quality completions
4. Repeat (Iterative RFT)

### Inference-Time Compute Scaling

BoN is a form of **test-time compute scaling** trading more compute at inference for better outputs:

| N (samples) | Relative compute | Typical reward improvement |
|------------|-----------------|---------------------------|
| 1 | 1x | Baseline |
| 4 | 4x | +0.5-1.0 reward units |
| 16 | 16x | +1.0-1.5 reward units |
| 64 | 64x | +1.5-2.0 reward units |
| 256 | 256x | +2.0-2.5 reward units |

Recent work (Snell et al., 2024) shows that **inference-time compute can substitute for
training-time compute**: a smaller model with large $N$ can match a larger model with $N=1$.


## Iterative DPO & Online vs Offline Alignment

### The Offline Limitation

Standard DPO is **offline**: it trains on a static preference dataset collected from some
behavior policy (typically the SFT model). This creates a **distribution shift** problem:
as the policy improves, it moves away from the region of data coverage, and further updates
are made on increasingly out-of-distribution examples.

### Iterative DPO

Iterative DPO (also called Online DPO) addresses distribution shift by alternating between:
1. **Data generation**: sample new completions from the current policy $\pi_\theta^{(t)}$
2. **Preference labeling**: score pairs using a reward model or AI feedback
3. **Policy update**: run DPO on the fresh preference pairs

```
Iteration t:
  For each prompt x in batch:
    y1, y2 ~ pi_theta^(t)(. | x)       # sample from current policy
    label = reward_model(x, y1, y2)     # determine preference
    add (x, y_chosen, y_rejected) to D_t
  pi_theta^(t+1) = DPO_update(pi_theta^(t), D_t)
```

### Related Online Methods

**WPO (Wei et al., 2024):** Weighted Preference Optimization adds importance weights to
offline DPO to correct for distribution shift without requiring online rollouts:

$$\mathcal{L}_{WPO} = -\mathbb{E}\left[w(x,y_w,y_l) \cdot \log \sigma\left(\beta(h_\theta(y_w) - h_\theta(y_l))\right)\right]$$

**RAFT (Reward-rAnked Fine-Tuning):** Generate rollouts, rank by reward, SFT on top-k.
Simple and effective; essentially online rejection sampling fine-tuning.

**NCA (Noise Contrastive Alignment):** Frames alignment as contrastive learning with
noise-based rejection. Avoids reference model by using negative sampling.

**RPO (Robust Preference Optimization):** Adds noise robustness to DPO by modeling
label noise in human preferences explicitly.

### Online vs Offline Tradeoffs

| Property | Offline DPO | Online/Iterative DPO |
|----------|-------------|---------------------|
| Data freshness | Static; may be OOD | Always on-policy |
| Compute cost | Low (no rollouts) | Higher (generation + scoring) |
| Reward model required | No (if using fixed dataset) | Yes (for online labeling) |
| Convergence | May stagnate | Continuously improves |
| Implementation complexity | Simple | Moderate to high |

The field is converging toward hybrid approaches: mostly offline training with periodic
online data refreshes, which balances cost and distribution coverage.


## Reward Hacking and the Alignment Tax

### Goodhart's Law in Alignment

> "When a measure becomes a target, it ceases to be a good measure."

In alignment, this manifests as **reward hacking**: the model discovers behaviors that score
high on the reward model but don't correspond to genuinely preferred outputs.

### Common Reward Hacking Patterns

| Pattern | Description | Example |
|---------|-------------|--------|
| Sycophancy | Agreeing with user even when wrong | "You're absolutely right..." |
| Verbosity | Longer = higher reward (reward model artifact) | Padding responses with filler |
| Format gaming | Specific formatting scores high | Always using bullet points/markdown |
| Hedging | Excessive uncertainty reduces perceived error | "It's complicated..." |
| Repetition | High-confidence phrases repeated | Saying "Certainly!" every response |

### The KL Penalty Tradeoff

The KL penalty $\beta$ directly controls the reward-hacking / alignment tradeoff:

$$J_{RLHF}(\pi_\theta) = \underbrace{r_\phi(x,y)}_{\text{reward}} - \underbrace{\beta \cdot \text{KL}[\pi_\theta \| \pi_{ref}]}_{\text{KL regularization}}$$

As $\beta \to 0$: model exploits the reward function (reward hacking)

As $\beta \to \infty$: model stays at the reference (no alignment signal)

The **optimal $\beta$** balances reward maximization with distribution preservation.
Empirically, $\beta \in [0.01, 0.5]$ works well across different model scales.

### The Alignment Tax

The **alignment tax** refers to measurable capability degradation after alignment training:

$$\text{Alignment Tax} = \text{Perf}_{pretrained} - \text{Perf}_{aligned}$$

on benchmarks that don't directly measure instruction-following (e.g., MMLU, HumanEval, GSM8K).

Observed effects:
- **Likelihood displacement**: DPO reduces log-probabilities of both chosen AND rejected responses
- **Knowledge loss**: heavy alignment can reduce recall of factual information
- **Format rigidity**: model over-generalizes instruction format to unrelated tasks

### Mitigation Strategies

| Strategy | Mechanism |
|----------|----------|
| Larger $\beta$ | Stronger KL constraint prevents large distribution shifts |
| Mix SFT + DPO | Add SFT loss on chosen responses during DPO training |
| Capability-preserving data | Include capability-focused examples in preference data |
| Model merging (SLERP/LERP) | Interpolate aligned and base model weights |
| Iterative alignment | Small alignment steps with capability evals between rounds |
| DPO-positive | Modify loss to prevent log-prob of chosen from decreasing |

Recent methods like **SimPO** and **ORPO** show reduced alignment tax compared to DPO,
suggesting that the exact formulation of the preference loss matters significantly for
preserving model capabilities.


In [3]:
# TRL Alignment Trainers: DPO, KTO, ORPO
# pip install trl transformers datasets accelerate peft bitsandbytes

TRL_CODE = '''
# ===================================================================
# 1. DPO Training with TRL
# ===================================================================
from trl import DPOTrainer, DPOConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# DPO dataset format: prompt + chosen + rejected
dpo_data = [
    {
        "prompt": "What is the capital of France?",
        "chosen": "The capital of France is Paris, a major European city on the Seine River.",
        "rejected": "France has many cities. I think it might be Lyon or maybe Marseille."
    },
    {
        "prompt": "Explain photosynthesis briefly.",
        "chosen": "Photosynthesis is the process by which plants convert sunlight, CO2, and water into glucose and oxygen using chlorophyll.",
        "rejected": "Plants eat sunlight. It involves some chemicals in leaves."
    },
]
dpo_dataset = Dataset.from_list(dpo_data)

dpo_config = DPOConfig(
    beta=0.1,                    # KL penalty coefficient
    max_length=512,
    max_prompt_length=256,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-7,
    bf16=True,
    output_dir="./dpo_output",
    logging_steps=10,
    loss_type="sigmoid",         # standard DPO; use "ipo" for IPO, "hinge" for SLiC
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,              # None = use frozen copy of model as reference
    args=dpo_config,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
)
dpo_trainer.train()
dpo_trainer.save_model("./dpo_output/final_model")


# ===================================================================
# 2. KTO Training with TRL
# ===================================================================
from trl import KTOTrainer, KTOConfig

# KTO dataset format: prompt + completion + label (True=good, False=bad)
# No pairing required each example is independent
kto_data = [
    {
        "prompt": "What is 2+2?",
        "completion": "2+2 equals 4.",
        "label": True           # desirable response
    },
    {
        "prompt": "What is 2+2?",
        "completion": "I am not sure, maybe 5 or 6?",
        "label": False          # undesirable response
    },
    {
        "prompt": "Write a haiku about rain.",
        "completion": "Drops fall on the earth / Quiet rhythm of the sky / Grass drinks deeply now",
        "label": True
    },
]
kto_dataset = Dataset.from_list(kto_data)

kto_config = KTOConfig(
    beta=0.1,
    desirable_weight=1.0,        # lambda_D: weight for desirable examples
    undesirable_weight=1.0,      # lambda_U: weight for undesirable (set > 1.0 for loss aversion)
    max_length=512,
    per_device_train_batch_size=2,
    num_train_epochs=1,
    learning_rate=5e-7,
    output_dir="./kto_output",
)

kto_trainer = KTOTrainer(
    model=model,
    ref_model=None,
    args=kto_config,
    train_dataset=kto_dataset,
    tokenizer=tokenizer,
)
kto_trainer.train()


# ===================================================================
# 3. ORPO Training with TRL
# ===================================================================
from trl import ORPOTrainer, ORPOConfig

# ORPO dataset format: same as DPO (prompt + chosen + rejected)
# But no reference model is used SFT + preference in a single pass
orpo_data = [
    {
        "prompt": "Translate to French: Hello, how are you?",
        "chosen": "Bonjour, comment allez-vous ?",
        "rejected": "Bonjour, como estas ?"
    },
    {
        "prompt": "What causes rainbows?",
        "chosen": "Rainbows form when sunlight refracts, disperses, and reflects inside water droplets, separating white light into its spectral colors.",
        "rejected": "Rainbows are caused by magic in the sky after rain."
    },
]
orpo_dataset = Dataset.from_list(orpo_data)

orpo_config = ORPOConfig(
    beta=0.1,                    # lambda in ORPO (weight of OR loss)
    max_length=512,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=8e-6,          # ORPO uses higher LR than DPO (no ref model)
    output_dir="./orpo_output",
)

orpo_trainer = ORPOTrainer(
    model=model,
    # No ref_model argument for ORPO!
    args=orpo_config,
    train_dataset=orpo_dataset,
    tokenizer=tokenizer,
)
orpo_trainer.train()
'''
print(TRL_CODE)



# ===================================================================
# 1. DPO Training with TRL
# ===================================================================
from trl import DPOTrainer, DPOConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# DPO dataset format: prompt + chosen + rejected
dpo_data = [
    {
        "prompt": "What is the capital of France?",
        "chosen": "The capital of France is Paris, a major European city on the Seine River.",
        "rejected": "France has many cities. I think it might be Lyon or maybe Marseille."
    },
    {
        "prompt": "Explain photosynthesis briefly.",
        "chosen": "Photosynthesis is the process by which pla

## Additional Learning Resources

### Foundational Papers

- [InstructGPT / RLHF](https://arxiv.org/abs/2203.02155) Ouyang et al., 2022. The seminal RLHF paper that trained GPT-3.5.
- [DPO](https://arxiv.org/abs/2305.18290) Rafailov et al., 2023. Direct Preference Optimization: Your Language Model is Secretly a Reward Model.
- [IPO](https://arxiv.org/abs/2310.12036) Azar et al., 2023. A General Theoretical Paradigm to Understand Learning from Human Feedback.
- [KTO](https://arxiv.org/abs/2402.01306) Ethayarajh et al., 2024. Model Alignment as Prospect Theoretic Optimization.
- [ORPO](https://arxiv.org/abs/2403.07691) Hong et al., 2024. ORPO: Monolithic Preference Optimization without Reference Model.
- [SimPO](https://arxiv.org/abs/2405.14734) Meng et al., 2024. Simple Preference Optimization with a Reference-Free Reward.
- [Constitutional AI](https://arxiv.org/abs/2212.08073) Bai et al., 2022. Constitutional AI: Harmlessness from AI Feedback.
- [SPIN](https://arxiv.org/abs/2401.01335) Chen et al., 2024. Self-Play Fine-Tuning Converts Weak Language Models to Strong Language Models.

### Tools & Libraries

- [TRL (Transformer Reinforcement Learning)](https://huggingface.co/docs/trl/) HuggingFace library with DPOTrainer, KTOTrainer, ORPOTrainer, PPOTrainer, and more.
- [Alignment Handbook](https://github.com/huggingface/alignment-handbook) HuggingFace recipes for reproducing state-of-the-art aligned models (Zephyr, Starling).
- [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF) High-performance RLHF training framework with Ray-based distributed training.
- [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory) Unified framework supporting 100+ LLMs with RLHF, DPO, KTO, ORPO.

### Courses & Tutorials

- [Deeplearning.ai RLHF Course](https://www.deeplearning.ai/short-courses/reinforcement-learning-from-human-feedback/) Practical RLHF with TRL and HuggingFace.
- [HuggingFace NLP Course Chapter on Alignment](https://huggingface.co/learn/nlp-course)
- [Anthropic's Alignment Research](https://www.anthropic.com/research) Papers and blog posts on Constitutional AI, RLHF scaling, and safety.
- [Lilian Weng's Blog: RLHF Overview](https://lilianweng.github.io/posts/2023-01-27-the-transformer-family-v2/) Thorough technical survey of alignment methods.
